# MOSS Transcription and Diarization Research

Process all recordings in `audio/active/` sequentially, then inspect individual results. Keep unused recordings in `audio/standby/`. This notebook reuses the project modules rather than duplicating inference or export logic.

## 1. Environment

Select the project **.venv/bin/python** kernel in VS Code. Install dependencies from a terminal in the project root:

```bash
export PATH="$PWD/.tools/bin:$PATH"
uv pip install --python .venv -r requirements-research.txt --torch-backend=auto
```

FFmpeg is provided by `imageio-ffmpeg` inside the project environment; a system installation is optional. The first inference downloads model weights; model loading executes the official remote model code. No model inference has been run in this notebook yet.


Configuration reviewed against upstream documentation: [model usage analysis](docs/model_usage_analysis.md). Current Typhoon uses 30-second windows, BF16 on CUDA, Thai language, and timestamps. Earlier timing reports used a different configuration and are historical; rerun for current measurements.

In [75]:
import importlib.util
import shutil
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "speech_to_text").is_dir():
    raise RuntimeError("Open the notebook from the project root.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project .venv/bin/python kernel before continuing.")
print("Python:", sys.executable)
for name in ("torch", "transformers", "moss_transcribe_diarize", "ipywidgets"):
    print(f"{name}: {'available' if importlib.util.find_spec(name) else 'missing'}")
from speech_to_text.outputs import resolve_ffmpeg
print("FFmpeg:", resolve_ffmpeg())


Python: /home/air-520/workspaces/AI-speech-to-text/.venv/bin/python
torch: available
transformers: available
moss_transcribe_diarize: available
ipywidgets: available
FFmpeg: /home/air-520/workspaces/AI-speech-to-text/.venv/lib/python3.10/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2


## 2. Discover active audio

Place recordings to process in `audio/active/`. Keep other recordings in `audio/standby/`; they are never scanned. Subfolders inside active are included. Run this cell again after moving files. The batch processes every discovered active recording in sorted order, without moving or deleting inputs.


In [76]:
from IPython.display import Audio, display

AUDIO_DIR = PROJECT_ROOT / "audio" / "active"
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus", ".wma"}
audio_files = sorted(
    (path for path in AUDIO_DIR.rglob("*")
     if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
     and not path.is_symlink()
     and path.resolve().is_relative_to(AUDIO_DIR.resolve())),
    key=lambda path: str(path.relative_to(AUDIO_DIR)).lower(),
)
print(f"Found {len(audio_files)} active audio files.")
for index, path in enumerate(audio_files):
    print(f"{index}: {path.relative_to(AUDIO_DIR)}")
if not audio_files:
    print("Move recordings from audio/standby to audio/active, then rerun this cell.")


Found 1 active audio files.
0: subway-test-02-thai_n_eng.wav


In [77]:
# Optional preview of one active input before running the full batch.
PREVIEW_INDEX = 0
if audio_files:
    print(audio_files[PREVIEW_INDEX].relative_to(PROJECT_ROOT))
    display(Audio(filename=str(audio_files[PREVIEW_INDEX])))


audio/active/subway-test-02-thai_n_eng.wav


## 3. Configure the experiment

Start with a short recording with known speakers. `auto` selects the available device; use `cpu` when needed. Increase the token limit if the transcript ends early. Large recordings can exceed available memory. Speaker IDs are model labels, not real names.

`LANGUAGES = ["Thai", "English"]` appends a language hint to the official MOSS prompt and requests original-language transcription, including language switches. It is a prompt hint, not a strict decoder language restriction. Set it to `None` to omit the hint.


In [78]:
import tempfile
from datetime import datetime, timezone

MODEL_ID = "OpenMOSS-Team/MOSS-Transcribe-Diarize"
DEVICE = "cuda:0"
MAX_NEW_TOKENS = 32768
LANGUAGES = ["Thai", "English"]  # Use None to keep the default prompt without language hints.

output_root = PROJECT_ROOT / "outputs"
output_root.mkdir(exist_ok=True)
batch_dir = Path(tempfile.mkdtemp(prefix="research-batch-", dir=output_root))
print("Batch directory:", batch_dir)

import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("This comparison requires two CUDA GPUs: MOSS on cuda:0 and Typhoon on cuda:1.")
for gpu_index in (0, 1):
    free, total = torch.cuda.mem_get_info(gpu_index)
    print(f"cuda:{gpu_index}: {torch.cuda.get_device_name(gpu_index)}; free {free / 2**30:.2f} GiB / {total / 2**30:.2f} GiB")


Batch directory: /home/air-520/workspaces/AI-speech-to-text/outputs/research-batch-xfnv8_zq
cuda:0: NVIDIA RTX 4000 Ada Generation; free 18.39 GiB / 19.55 GiB
cuda:1: NVIDIA RTX 4000 Ada Generation; free 18.43 GiB / 19.55 GiB


## 4. Process every active recording

Run transcription and export sequentially for all discovered active files. Each input gets a separate result directory and experiment metadata. A failed file is recorded in `batch_summary.json` and processing continues with the next file. The existing inference adapter reloads the model for each recording. Inputs stay in active, so running this cell again processes them again.


In [79]:
import json
from time import perf_counter
import importlib
import inspect
import traceback
import speech_to_text.backend as backend
import speech_to_text.outputs as output_module

# Refresh cached modules when this cell is rerun in an existing kernel.
importlib.invalidate_caches()
backend = importlib.reload(backend)
output_module = importlib.reload(output_module)
transcribe = backend.transcribe
export_results = output_module.export_results
resolve_ffmpeg = output_module.resolve_ffmpeg
print("Backend:", backend.__file__, flush=True)
print("Transcribe signature:", inspect.signature(transcribe), flush=True)

if not audio_files:
    raise ValueError("No active recordings. Add files to audio/active and rerun discovery.")
if MAX_NEW_TOKENS <= 0:
    raise ValueError("MAX_NEW_TOKENS must be positive.")
resolve_ffmpeg()
results = []
batch_summary = []
for index, audio_path in enumerate(audio_files):
    run_dir = Path(tempfile.mkdtemp(prefix=f"{index:03d}-", dir=batch_dir))
    experiment = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "source": str(audio_path), "model": MODEL_ID, "device": DEVICE,
        "max_new_tokens": MAX_NEW_TOKENS, "languages": LANGUAGES, "python": sys.version,
        "output_dir": str(run_dir),
    }
    print(f"[{index + 1}/{len(audio_files)}] {audio_path.name}", flush=True)
    started = perf_counter()
    try:
        raw, segments = transcribe(audio_path, MODEL_ID, DEVICE, MAX_NEW_TOKENS, languages=LANGUAGES)
        (run_dir / "raw_transcript.txt").write_text(raw, encoding="utf-8")
        export_results(audio_path, run_dir, raw, segments)
        experiment.update(status="success", segment_count=len(segments))
        results.append({"source": audio_path, "run_dir": run_dir, "raw": raw, "segments": segments})
    except Exception as error:
        experiment.update(status="failed", error=f"{type(error).__name__}: {error}")
        print("Failed:", experiment["error"], flush=True)
        traceback.print_exc()
    experiment["elapsed_seconds"] = perf_counter() - started
    (run_dir / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")
    if experiment["status"] == "success":
        print(f"Success: {len(segments)} segments in {experiment['elapsed_seconds']:.2f}s; output: {run_dir}", flush=True)
    batch_summary.append(experiment)
    (batch_dir / "batch_summary.json").write_text(json.dumps(batch_summary, indent=2), encoding="utf-8")
print(f"Completed: {len(results)} successful, {len(batch_summary) - len(results)} failed.")

failed_count = len(batch_summary) - len(results)
if failed_count:
    raise RuntimeError(f"{failed_count} recordings failed. Inspect the tracebacks and {batch_dir / 'batch_summary.json'}.")


Backend: /home/air-520/workspaces/AI-speech-to-text/speech_to_text/backend.py
Transcribe signature: (audio_path, model_id, device_name, max_new_tokens, languages=None)
[1/1] subway-test-02-thai_n_eng.wav


Loading weights:   0%|          | 0/683 [00:00<?, ?it/s]

Success: 30 segments in 26.25s; output: /home/air-520/workspaces/AI-speech-to-text/outputs/research-batch-xfnv8_zq/000-xn0zgucl
Completed: 1 successful, 0 failed.


## 5. Inspect speaker segments

Inspect the full segment table and summed segment duration per speaker. Summed durations can double-count overlapping intervals and do not measure diarization accuracy. Compare the labels and words against the original audio.


In [80]:
# Select a successful result to inspect. Change the index and rerun this cell.
REVIEW_INDEX = 0
if not results:
    raise ValueError("No successful results. Inspect batch_summary.json for errors.")
for index, result in enumerate(results):
    print(f"{index}: {result['source'].name}")
selected_result = results[REVIEW_INDEX]
audio_path = selected_result["source"]
run_dir = selected_result["run_dir"]
raw = selected_result["raw"]
segments = selected_result["segments"]
print("Reviewing:", audio_path.name)

from collections import defaultdict
from html import escape
from IPython.display import HTML

rows = "".join(
    f"<tr><td>{s.start:.2f}</td><td>{s.end:.2f}</td>"
    f"<td>{escape(s.speaker)}</td><td>{escape(s.text)}</td></tr>"
    for s in segments
)
display(HTML("<table><tr><th>Start (s)</th><th>End (s)</th><th>Speaker</th><th>Text</th></tr>" + rows + "</table>"))
speaker_seconds = defaultdict(float)
for segment in segments:
    speaker_seconds[segment.speaker] += segment.end - segment.start
print("Summed segment seconds:", dict(speaker_seconds))
if not segments:
    print("No parsed segments. Inspect the raw response before exporting.")


0: subway-test-02-thai_n_eng.wav
Reviewing: subway-test-02-thai_n_eng.wav


Start (s),End (s),Speaker,Text
2.12,3.12,S01,คั่นชีตได้ไหมคะ
3.12,3.84,S02,ได้ค่ะ
4.72,5.62,S01,เป็นเมนูไหนคะ
6.54,9.52,S02,เอ่อ เดี๋ยวเดี๋ยวเดี๋ยวข้างไก่เธอเธอจะเก็บอะไรคะ
9.52,13.88,S01,เป็นไก่บาร์บิคิว บูกโอกิ ไก่ลมควัน แล้วก็ไก่อกค่ะ
14.34,15.24,S02,ของเลยค่ะ
15.42,16.02,S02,ดูไหม
16.18,17.12,S01,ได้ค่ะ
19.78,21.78,S01,เด็กคริสต์มีส์ ยูเพลย์ทูเกตเตอร์
23.16,24.12,S01,อันนี้เมนูไหนคะ


Summed segment seconds: {'S01': 38.02000000000003, 'S02': 7.199999999999999}


## 6. Listen to exported speaker audio

The batch already exported TXT, JSON, and WAV files. Listen to the result selected above. Speaker WAVs concatenate that speaker's intervals; simultaneous voices remain audible. This does not perform source separation.


In [81]:
for speaker_file in sorted(run_dir.glob("S*.wav")):
    print(speaker_file.name)
    display(Audio(filename=str(speaker_file)))
print("Saved results:", run_dir)


S01.wav


S02.wav


Saved results: /home/air-520/workspaces/AI-speech-to-text/outputs/research-batch-xfnv8_zq/000-xn0zgucl


## 7. Research notes

Record the language, number of known speakers, missed or incorrect words, speaker switches, timestamp errors, overlapping speech, and whether the raw transcript appears truncated. These notes are manual observations, not measured WER or DER scores. Run this cell to save them with the experiment.


In [82]:
NOTES = """Language:
Known speaker count:
Transcription observations:
Speaker attribution observations:
Timestamp observations:
Overlapping speech:
Potential truncation:
Next experiment:
"""
(run_dir / "notes.md").write_text(NOTES, encoding="utf-8")
print("Saved research notes.")


Saved research notes.


## 8. Compare MOSS and Typhoon on the same recordings

MOSS runs on `cuda:0`; Typhoon Whisper runs on `cuda:1`. Both receive the same original audio. The passes run sequentially, each pinned to its own GPU. Typhoon is a Thai-focused ASR model and does not provide speaker IDs. MOSS speaker WAVs are not used as Typhoon inputs.

Typhoon defaults to `TYPHOON_LANGUAGE = "thai"` following its model card. Set `TYPHOON_LANGUAGE = None` for a separate automatic-detection experiment. Unlike MOSS prompt hints, Whisper accepts one language token, not a list of languages. English and code-switching quality need manual evaluation.

Timings include loading, preprocessing, inference, and (for MOSS) speaker export; initial downloads also affect elapsed time. They are workflow timings, not a controlled speed benchmark. Side-by-side differences are not accuracy scores: a human reference transcript is required for CER/WER evaluation.

Source: [Typhoon model card](https://huggingface.co/typhoon-ai/typhoon-whisper-large-v3).

Typhoon uses 30-second overlapping windows with a five-second stride on each side, timestamps, BF16 on GPU, batch size 1, and greedy decoding (`num_beams=1`) to limit memory on GPU 1.

By default the entire Typhoon model runs on GPU 1 (`TYPHOON_CPU_OFFLOAD = False`). Optional `True` keeps the FP32 encoder on CPU and the BF16 decoder on `cuda:1` if GPU memory is limited. Actual encoder placement and precision are recorded in comparison metadata.


In [83]:
TYPHOON_MODEL_ID = "typhoon-ai/typhoon-whisper-large-v3"
TYPHOON_DEVICE = "cuda:1"
TYPHOON_LANGUAGE = "thai"
TYPHOON_CPU_OFFLOAD = False  # Run the entire Typhoon model on GPU 1.

import speech_to_text.typhoon as typhoon_backend
typhoon_backend = importlib.reload(typhoon_backend)
comparison_results = []
comparison_failures = []
for result in results:
    source = result["source"]
    directory = result["run_dir"]
    print(f"Typhoon on {TYPHOON_DEVICE}: {source.name}", flush=True)
    started = perf_counter()
    try:
        typhoon_result = typhoon_backend.transcribe_typhoon(
            source, TYPHOON_MODEL_ID, TYPHOON_DEVICE, language=TYPHOON_LANGUAGE, cpu_offload=TYPHOON_CPU_OFFLOAD,
        )
        elapsed = perf_counter() - started
        moss_metadata = json.loads((directory / "experiment.json").read_text())
        comparison = {
            "source": str(source),
            "moss": {
                "model": MODEL_ID, "device": DEVICE, "languages": LANGUAGES,
                "text": " ".join(segment.text for segment in result["segments"]),
                "elapsed_seconds": moss_metadata["elapsed_seconds"],
                "segment_count": len(result["segments"]),
                "speaker_count": len({segment.speaker for segment in result["segments"]}),
            },
            "typhoon": {
                "model": TYPHOON_MODEL_ID, "language": TYPHOON_LANGUAGE,
                **typhoon_result, "elapsed_seconds": elapsed,
            },
        }
        (directory / "typhoon_transcript.txt").write_text(typhoon_result["text"], encoding="utf-8")
        (directory / "comparison.json").write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
        comparison_results.append(comparison)
        print(f"Success: {elapsed:.2f}s; output: {directory / 'comparison.json'}", flush=True)
    except Exception as error:
        comparison_failures.append({"source": str(source), "error": f"{type(error).__name__}: {error}"})
        traceback.print_exc()
(batch_dir / "comparison_summary.json").write_text(
    json.dumps({"results": comparison_results, "failures": comparison_failures}, ensure_ascii=False, indent=2), encoding="utf-8",
)
if comparison_failures:
    raise RuntimeError(f"Typhoon failed for {len(comparison_failures)} recordings; inspect comparison_summary.json.")


Typhoon on cuda:1: subway-test-02-thai_n_eng.wav


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Success: 12.03s; output: /home/air-520/workspaces/AI-speech-to-text/outputs/research-batch-xfnv8_zq/000-xn0zgucl/comparison.json


## 9. Review model differences

Compare the original-language text and listen to the source recording before deciding which output is more accurate. MOSS alone supplies speaker labels. No WER or CER is reported without a reference transcript. The HTML report highlights textual differences at character level, including Thai text without spaces.


In [84]:
from difflib import HtmlDiff

comparison_style = """
<style>
.model-comparison { background: #f8fafc; color: #263445; padding: 24px; border: 1px solid #d8dfe7; border-radius: 12px; font: 15px/1.8 system-ui, sans-serif; color-scheme: light; }
.model-comparison h2, .model-comparison h3 { color: #263445; margin: 0 0 12px; }
.model-comparison p { color: #526174; }
.model-comparison .transcripts { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 20px; margin: 20px 0; }
.model-comparison article { background: #fff; border: 1px solid #d8dfe7; border-radius: 8px; padding: 18px; overflow-wrap: anywhere; }
.model-comparison .speech { white-space: pre-wrap; line-height: 2; font-size: 16px; }
.model-comparison table { border-collapse: collapse; width: 100%; background: #fff; color: #263445; font: inherit; }
.model-comparison th, .model-comparison td { padding: 10px 12px; border: 1px solid #d8dfe7; text-align: left; vertical-align: top; color: #263445; background: #fff; }
.model-comparison th, .model-comparison .diff_header { background: #edf1f5; color: #425267; font-weight: 600; }
.model-comparison .diff-wrapper { overflow-x: auto; margin-top: 12px; }
.model-comparison table.diff { table-layout: fixed; min-width: 680px; }
.model-comparison table.diff td { white-space: normal !important; overflow-wrap: anywhere; line-height: 2; }
.model-comparison table.diff .diff_next { display: none; }
.model-comparison table.diff td.diff_header { width: 28px; font-size: 12px; }
.model-comparison .diff_add { background: #e5eee8; color: #274333; text-decoration: underline; text-decoration-color: #8ca694; text-underline-offset: 3px; }
.model-comparison .diff_sub { background: #f0e6e3; color: #65483f; text-decoration: line-through; text-decoration-color: #b49a91; }
.model-comparison .diff_chg { background: #e4eaf2; color: #304762; border-bottom: 1px solid #9bacc0; }
.model-comparison .legend { font-size: 13px; color: #526174; }
@media (max-width: 760px) { .model-comparison .transcripts { grid-template-columns: 1fr; } }
</style>
"""
for comparison in comparison_results:
    moss = comparison["moss"]
    typhoon = comparison["typhoon"]
    title = escape(Path(comparison["source"]).name)
    metrics = (
        "<table><tr><th>Model</th><th>Device</th><th>Workflow seconds</th><th>Speaker count</th></tr>"
        f"<tr><td>MOSS</td><td>{escape(moss['device'])}</td><td>{moss['elapsed_seconds']:.2f}</td><td>{moss['speaker_count']}</td></tr>"
        f"<tr><td>Typhoon</td><td>{escape(typhoon['device'])} (encoder: {escape(typhoon['encoder_device'])})</td>"
        f"<td>{typhoon['elapsed_seconds']:.2f}</td><td>Not provided</td></tr></table>"
    )
    transcripts = (
        '<div class="transcripts">'
        f'<article><h3>MOSS</h3><div class="speech">{escape(moss["text"])}</div></article>'
        f'<article><h3>Typhoon</h3><div class="speech">{escape(typhoon["text"])}</div></article></div>'
    )
    difference = HtmlDiff(wrapcolumn=65).make_table(
        [moss["text"]], [typhoon["text"]], fromdesc="MOSS", todesc="Typhoon",
    )
    report = (
        comparison_style + f'<section class="model-comparison"><h2>{title}</h2>'
        + metrics + transcripts + '<h3>Text differences</h3>'
        + '<p class="legend">Soft blue: changed text. Underline: added text. Strikethrough: removed text. Differences do not measure accuracy.</p>'
        + '<div class="diff-wrapper">' + difference + '</div></section>'
    )
    directory = next(item["run_dir"] for item in results if str(item["source"]) == comparison["source"])
    document = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Model comparison</title></head><body style="margin:24px;background:#eef1f5">' + report + '</body></html>'
    (directory / "comparison.html").write_text(document, encoding="utf-8")
    display(HTML(report))
